# HIP-LLM: StrategyQA failure probability under an operational profile

This notebook performs an end-to-end **labelled benchmark evaluation**:

1. loads the labelled StrategyQA development split from the official StrategyQA repository;
2. sends each question independently to a pinned OpenAI model snapshot;
3. parses the model's `YES`/`NO` answer deterministically;
4. compares it with the gold answer;
5. defines an explicit operational profile over question-complexity strata; and
6. estimates the operational probability of failure with HIP-LLM's hierarchical imprecise-Bayesian model.

This is different from the repository's legacy `FailureProb` token-logprob score.  
`1 - token confidence` is a prompt-level heuristic; it does not use an operational profile.  
The calculation below uses **observed correctness labels plus explicit workload weights**.

> **Cost:** API calls are made for `N_QUESTIONS` items. Start with a small value, inspect the pipeline, and then increase it.

In [ ]:
# Install the current repository and the official OpenAI Python SDK.
# After installation, Colab may ask you to restart the runtime only if an old
# incompatible package version was already loaded.
%pip install -q "git+https://github.com/koo-ec/HIP_LLM.git@main" "openai>=1.40" pandas numpy matplotlib

In [ ]:
import json
import os
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from openai import OpenAI

from HIPLLM import (
    OperationalFailureProb,
    decomposition_stratum,
    load_strategyqa,
    parse_strategyqa_answer,
    quick_inference_settings,
)

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not present in the notebook environment. "
        "Set it in Colab Secrets or os.environ before running this cell."
    )

print("Imports and API-key check completed.")

## Reproducible configuration

- The StrategyQA repository revision is pinned to a commit.
- The OpenAI model is a pinned snapshot rather than a moving alias.
- Every question is sent as an independent request with no conversational history.
- API failures stop the run; they are not silently converted into benchmark failures.
- Unparseable model text is explicitly counted as an answer failure.

In [ ]:
MODEL = "gpt-4.1-mini-2025-04-14"
STRATEGYQA_REVISION = "1ba1e97452e189569357876f2854b01357ffbe37"

N_QUESTIONS = 30       # Increase after the small run works.
SEED = 7
MAX_OUTPUT_TOKENS = 8
REQUEST_DELAY_SECONDS = 0.05

# "dataset_proportional" uses the full labelled dev-set stratum distribution.
# "custom" uses the deployment profile supplied below.
OP_MODE = "dataset_proportional"
CUSTOM_OPERATIONAL_PROFILE = {
    "short": 0.20,
    "medium": 0.45,
    "long": 0.35,
}

PREDICTION_CACHE = Path("/content/strategyqa_openai_predictions.csv")
SUMMARY_FILE = Path("/content/strategyqa_operational_failure_summary.json")

SYSTEM_INSTRUCTION = (
    "You are being evaluated on StrategyQA. "
    "Answer the question with exactly one word: YES or NO. "
    "Do not provide an explanation."
)

if N_QUESTIONS < 3:
    raise ValueError("N_QUESTIONS must be at least 3 so every stratum can be sampled.")

## Load StrategyQA and define transparent workload strata

The official dataset does not define `short`, `medium`, and `long` operational categories.  
Here they are a documented workload design based on the number of gold decomposition steps:

- **short:** at most 2 steps;
- **medium:** exactly 3 steps;
- **long:** 4 or more steps.

For a real application, replace these categories and weights with a profile derived from deployment logs, requirements, or expert elicitation.

In [ ]:
rows = load_strategyqa(
    "dev",
    revision=STRATEGYQA_REVISION,
)

all_items = pd.DataFrame(
    {
        "qid": [str(row["qid"]) for row in rows],
        "question": [row["question"] for row in rows],
        "gold_answer": [bool(row["answer"]) for row in rows],
        "decomposition_steps": [len(row["decomposition"]) for row in rows],
        "stratum": [decomposition_stratum(row) for row in rows],
    }
)

STRATA = ["short", "medium", "long"]
available_counts = all_items["stratum"].value_counts().reindex(STRATA, fill_value=0)
if (available_counts == 0).any():
    raise RuntimeError(
        f"At least one operational stratum has no dataset items: {available_counts.to_dict()}"
    )

display(all_items.head())
print("Full dev-set stratum counts:", available_counts.to_dict())

In [ ]:
def stratified_sample(
    frame: pd.DataFrame,
    n: int,
    labels: list[str],
    seed: int,
) -> pd.DataFrame:
    """Select at least one item per stratum, then fill the remaining quota."""
    if n > len(frame):
        n = len(frame)
    if n < len(labels):
        raise ValueError("Sample size is smaller than the number of strata.")

    selected_indices: list[int] = []
    for offset, label in enumerate(labels):
        group = frame[frame["stratum"] == label]
        selected_indices.extend(
            group.sample(n=1, random_state=seed + offset).index.tolist()
        )

    remaining = n - len(selected_indices)
    if remaining:
        pool = frame.drop(index=selected_indices)
        selected_indices.extend(
            pool.sample(n=remaining, random_state=seed + 100).index.tolist()
        )

    return (
        frame.loc[selected_indices]
        .sample(frac=1.0, random_state=seed + 200)
        .reset_index(drop=True)
    )


items = stratified_sample(all_items, N_QUESTIONS, STRATA, SEED)
sample_counts = items["stratum"].value_counts().reindex(STRATA, fill_value=0)

if OP_MODE == "dataset_proportional":
    operational_profile = {
        label: float(available_counts[label] / available_counts.sum())
        for label in STRATA
    }
elif OP_MODE == "custom":
    operational_profile = {label: float(CUSTOM_OPERATIONAL_PROFILE[label]) for label in STRATA}
else:
    raise ValueError("OP_MODE must be 'dataset_proportional' or 'custom'.")

if not np.isclose(sum(operational_profile.values()), 1.0, atol=1e-9):
    raise ValueError(f"Operational-profile weights must sum to 1: {operational_profile}")
if any(weight < 0 for weight in operational_profile.values()):
    raise ValueError("Operational-profile weights must be non-negative.")
if any(sample_counts[label] == 0 for label in operational_profile):
    raise RuntimeError(
        "The selected benchmark sample has no observations for an OP stratum: "
        f"{sample_counts.to_dict()}"
    )

print("Selected sample counts:", sample_counts.to_dict())
print("Operational profile:", operational_profile)
display(items.head())

## Run or resume OpenAI predictions

The cache is updated after every successful request. Re-running the cell resumes completed question IDs when the cached model and dataset revision match the current configuration.

In [ ]:
client = OpenAI(timeout=60.0, max_retries=2)

required_cache_columns = {
    "qid",
    "model_requested",
    "dataset_revision",
    "question",
    "gold_answer",
    "stratum",
    "raw_response",
    "parsed_answer",
    "parseable",
    "correct",
    "response_id",
    "model_resolved",
    "input_tokens",
    "output_tokens",
}

if PREDICTION_CACHE.exists():
    cached = pd.read_csv(PREDICTION_CACHE)
    if not required_cache_columns.issubset(cached.columns):
        print("Ignoring an incompatible cache schema.")
        cached = pd.DataFrame(columns=sorted(required_cache_columns))
    else:
        cached = cached[
            (cached["model_requested"] == MODEL)
            & (cached["dataset_revision"] == STRATEGYQA_REVISION)
        ].copy()
else:
    cached = pd.DataFrame(columns=sorted(required_cache_columns))

records = cached.to_dict("records")
completed = set(cached["qid"].astype(str)) if not cached.empty else set()

for position, row in items.iterrows():
    qid = str(row["qid"])
    if qid in completed:
        continue

    try:
        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_INSTRUCTION,
            input=row["question"],
            temperature=0,
            max_output_tokens=MAX_OUTPUT_TOKENS,
        )
    except Exception as exc:
        raise RuntimeError(
            f"OpenAI request failed for qid={qid}. "
            "The run stops so an API error is not silently treated as an answer failure."
        ) from exc

    raw_text = (response.output_text or "").strip()
    parsed = parse_strategyqa_answer(raw_text)
    parseable = parsed is not None

    # Explicit service-level policy for textual output:
    # an unparseable answer is counted as a failed answer.
    correct = bool(parseable and parsed == bool(row["gold_answer"]))

    usage = getattr(response, "usage", None)
    record = {
        "qid": qid,
        "model_requested": MODEL,
        "dataset_revision": STRATEGYQA_REVISION,
        "question": row["question"],
        "gold_answer": bool(row["gold_answer"]),
        "stratum": row["stratum"],
        "decomposition_steps": int(row["decomposition_steps"]),
        "raw_response": raw_text,
        "parsed_answer": parsed,
        "parseable": parseable,
        "correct": correct,
        "response_id": getattr(response, "id", None),
        "model_resolved": getattr(response, "model", None),
        "input_tokens": getattr(usage, "input_tokens", None) if usage else None,
        "output_tokens": getattr(usage, "output_tokens", None) if usage else None,
    }
    records.append(record)
    completed.add(qid)
    pd.DataFrame(records).to_csv(PREDICTION_CACHE, index=False)

    print(
        f"[{len(completed):>3}/{len(items)}] {qid}: "
        f"gold={bool(row['gold_answer'])}, parsed={parsed}, correct={correct}"
    )
    time.sleep(REQUEST_DELAY_SECONDS)

predictions = pd.DataFrame(records)
predictions = predictions[predictions["qid"].astype(str).isin(items["qid"].astype(str))].copy()
predictions = predictions.drop_duplicates(subset=["qid"], keep="last")
predictions = items.merge(
    predictions.drop(columns=["question", "gold_answer", "stratum", "decomposition_steps"]),
    on="qid",
    how="left",
    validate="one_to_one",
)

if predictions["correct"].isna().any():
    missing = predictions.loc[predictions["correct"].isna(), "qid"].tolist()
    raise RuntimeError(f"Missing predictions after the API run: {missing}")

predictions["correct"] = predictions["correct"].astype(bool)
predictions["parseable"] = predictions["parseable"].astype(bool)

print(f"Overall accuracy: {predictions['correct'].mean():.3f}")
print(f"Parseable-output rate: {predictions['parseable'].mean():.3f}")
display(predictions.head())

In [ ]:
stratum_results = (
    predictions.groupby("stratum", observed=True)
    .agg(
        questions=("qid", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        parseable_rate=("parseable", "mean"),
    )
    .reindex(STRATA)
)
stratum_results["empirical_failure_probability"] = 1.0 - stratum_results["accuracy"]
stratum_results["operational_weight"] = pd.Series(operational_profile)

display(stratum_results)

## Fit the HIP-LLM operational-profile model

For each stratum, the observed correct-answer count is treated as binomial evidence. HIP-LLM samples the hierarchical posterior across an admissible set of hyper-hyperparameters and aggregates stratum reliability using the explicit operational-profile weights.

The result is an **imprecise posterior family**. Therefore, the summary reports lower and upper bounds across admissible configurations rather than hiding the prior imprecision behind one value.

In [ ]:
settings = quick_inference_settings(
    seed=SEED,
    samples=2000,
    configurations=64,
)

estimator = OperationalFailureProb(
    profile=operational_profile,
    settings=settings,
    credible_level=0.95,
)

op_result = estimator.fit(
    outcomes=predictions["correct"].astype(int).tolist(),
    strata=predictions["stratum"].tolist(),
    domain_name="StrategyQA_decomposition_complexity",
)

summary = op_result.summary()
display(op_result.to_df())

summary_for_display = {
    key: value
    for key, value in summary.items()
    if key not in {"metadata", "operational_profile"}
}
display(pd.Series(summary_for_display, name="value").to_frame())

print("Operational profile:", summary["operational_profile"])
print("Inference metadata:", summary["metadata"])

### Interpretation

- `empirical_operational_failure_probability` is the direct weighted error rate using the observed stratum error rates.
- `posterior_expected_failure_lower` and `posterior_expected_failure_upper` bound the posterior mean failure probability across admissible prior configurations.
- `posterior_credible_lower` and `posterior_credible_upper` form an outer equal-tail credible envelope across those configurations.
- These are **workload-level** estimates. They are not confidence scores for an individual answer.
- The result is only representative of deployment when the operational profile and benchmark strata represent the intended workload.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(op_result.failure_samples.ravel(), bins=40)
plt.xlabel("Operational failure probability")
plt.ylabel("Posterior sample count")
plt.title("HIP-LLM posterior samples across admissible configurations")
plt.tight_layout()
plt.show()

In [ ]:
plot_frame = op_result.to_df().set_index("stratum")
plt.figure(figsize=(7, 4.5))
plt.bar(plot_frame.index, plot_frame["empirical_failure_probability"])
plt.xlabel("Operational stratum")
plt.ylabel("Observed failure probability")
plt.ylim(0, 1)
plt.title("Observed StrategyQA failure by workload stratum")
plt.tight_layout()
plt.show()

## Save reproducible outputs

The prediction CSV contains the question IDs, model snapshot, dataset revision, raw model text, parsed answers, gold answers, strata, and correctness. The JSON file records the operational profile and posterior summary.

In [ ]:
predictions.to_csv(PREDICTION_CACHE, index=False)

serialisable_summary = {
    **summary,
    "model_requested": MODEL,
    "dataset_revision": STRATEGYQA_REVISION,
    "n_questions": int(len(predictions)),
    "unparseable_policy": "count_as_answer_failure",
    "api_error_policy": "abort_run",
    "stratum_definition": {
        "short": "decomposition_steps <= 2",
        "medium": "decomposition_steps == 3",
        "long": "decomposition_steps >= 4",
    },
}
SUMMARY_FILE.write_text(
    json.dumps(serialisable_summary, indent=2, default=float),
    encoding="utf-8",
)

print("Predictions:", PREDICTION_CACHE)
print("Summary:", SUMMARY_FILE)